<!-- colab-badge -->
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RemusTeodorescu/DL-for-Engineers-Public-Course/blob/main/exercises/Ex09.1-burgers-2d/Ex09.1_02_control_panel.ipynb)

*Open this notebook in Google Colab. Its first code cell fetches the set's library files from the public course repository, so nothing needs uploading.*

<!-- course-header v3 -->

**Deep Learning for Engineering** · MSc, Aalborg University · 2026

Developed by **Remus Teodorescu** (ret@et.aau.dk), with support from Research Assistant **Noman Khan** (nomank@energy.aau.dk).

*Reference texts — read for the theory. Used as an **inspirational source** for this course, not as a source of its code:*

- Liu, *PINN with Python*, 2025.
- Prince, *Understanding Deep Learning*, MIT Press 2023.

Every notebook in this course has been **written and rewritten by the authors named above**. The code, the problems, the data and the exposition are **original to this course** and are not derived from any publisher's code listings or companion notebooks. Where notation matches a textbook's it is the standard notation of the field, and where an idea is a named author's it is cited as theirs in the text.

See `docs/PROVENANCE.md` for what each reference is cited for, set by set.

---

# Ex_09.1 · Notebook 02 — the control panel

**Paired with L9.1 · Laminar Flow**

Change parameters with sliders instead of editing code. Every run is recorded
in `runs`, and notebook 04 turns that list into a report.

**Paste your `residual_fn` and `loss_fn_factory` from notebook 01 into the
cell below the setup cell** — they are the physics, and they stay yours.

---

## 0 · Setup

In [ ]:
# files-cell v1 ----------------------------------------------------------
# This set's library files must sit beside the notebook. Locally they
# already do. On Google Colab, where a notebook opens on its own, they are
# fetched from the public course repository. Run this cell first.
import os, urllib.request
FILES = ['course_core.py', 'pinn_core.py', 'problem.py']
URL = "https://raw.githubusercontent.com/RemusTeodorescu/DL-for-Engineers-Public-Course/main/exercises/Ex09.1-burgers-2d/"
for f in FILES:
    if not os.path.exists(f):
        urllib.request.urlretrieve(URL + f, f)
        print("fetched", f)
print("files ready:", ", ".join(FILES))


In [ ]:
# --- setup: every Part 2 notebook opens with this cell ------------------
# Needs course_core.py, pinn_core.py and problem.py beside this notebook.
# On Colab the files cell above fetched them from the public course repository.
import os
for f in ("course_core.py", "pinn_core.py", "problem.py"):
    assert os.path.exists(f), f"{f} is missing - run the files cell above first"

from pinn_core import *                                  # noqa: F401,F403
import problem as pb
import numpy as np, torch, matplotlib.pyplot as plt

set_seed(88)
print("device:", DEVICE, " dtype:", torch.get_default_dtype())

In [ ]:
# --- paste your residual_fn and loss_fn_factory from notebook 01 here ---
raise NotImplementedError("paste your residual_fn and loss_fn_factory")

## 1 · The panel

In [ ]:
runs = []

def on_run(cfg):
    r = pb.run_study(cfg, residual_fn, loss_fn_factory)
    runs.append(r)
    plot_curves(r["history"], title=f"run {len(runs)}: {cfg}")
    plt.show()
    pb.plot_fields(r, t=0.5)
    print(f"\n{len(runs)} run(s) recorded")

panel = pb.control_panel(on_run)

## 2 · What to investigate

The panel exposes the parameters that matter. Suggested studies:

| Study | Vary | Hold fixed | Look for |
|---|---|---|---|
| Sampling | collocation points 500 → 20 000 | everything else | where the error plateaus |
| Capacity | neurons, layers | N_f | whether N_f/P\* stays above ~10 |
| Architecture | shared vs separate nets | parameter count | which wins, and by how much |
| Optimiser | L-BFGS epochs 0 → 800 | everything else | how much the handoff buys |

The readout under the sliders shows the viscosity and front width the Reynolds
slider implies, then two measures of the network's size: P\* — the neuron
count of L7.1, which the sampling condition is stated in — and the trainable
parameter count that `describe` reports. It warns when N_f/P\* falls below 10.

Change one thing at a time, and record the run before you change the next.

In [ ]:
if runs:
    pb.plot_error_vs_time(runs, [f"run {i+1}: {r['config']}"
                                 for i, r in enumerate(runs)])

## 3 · Save

In [ ]:
import pickle

os.makedirs(pb.OUTPUT_DIR, exist_ok=True)
path = os.path.join(pb.OUTPUT_DIR, "nb02_runs.pkl")
with open(path, "wb") as f:
    pickle.dump([{k: v for k, v in r.items() if k != "model"} for r in runs], f)
print(f"wrote {path}  ({len(runs)} runs)")